# H: Baseline vs responsible, with the retry-first rule

Notebook G compared plain Llama with our pipeline under today's oversight rule (`disagree_escalates`): when the two
reviewers disagree, the answer goes to a person at once. Only 3 of 50 answers were approved, and 44% went to a person.

This notebook runs **the same 50 baseline answers** through the pipeline again with the other rule, `retry_first`:

| Rule | Reviewers disagree in round 1 or 2 | Still disagree after round 3 |
|---|---|---|
| `disagree_escalates` (notebook G) | to a person at once | – |
| `retry_first` (this notebook) | retry with the reviewers' feedback | to a person |

Everything else is identical: same probes, same baseline answers (read from `base_answers.csv`, saved by notebook G),
same models and decoding. So the only difference between the two responsible columns is the oversight rule.

New here: the second detector also scores the rewriter's **last attempt**, approved or not, so we can see what the
rewriting achieves even when oversight holds it back.

**Before you start:** T4 GPU runtime; `HF_TOKEN` in Colab Secrets; notebook G already run (it saved the baseline
answers). Time: about 1.5–2 hours (more answers get up to 3 rounds). Progress is saved after every probe to
`responsible_rows_retry_first.csv`; notebook G's results in `responsible_rows.csv` are not touched.

In [ ]:
#  TEAM SETUP CELL
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/iibrohim-hash/debias.git"
BRANCH = "main"  # use your feature branch while developing, e.g. "iroda/detector"

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/debias")
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/debias")
else:  # local Jupyter: notebook is inside repo/notebooks
    REPO_DIR = Path.cwd().parent if (Path.cwd().parent / "src" / "config.py").exists() else Path.cwd()

if IN_COLAB or IN_KAGGLE:
    if not (REPO_DIR / "src").exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

try:  # autoreload is broken on some Colab images (Python 3.13: no 'imp' module)
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    print("autoreload not available: restart the runtime after editing src/")

from src import config, utils, contract
utils.set_seed(config.SEED)
if utils.get_hf_token():  # from Colab/Kaggle secrets; never type a token here (public repo)
    os.environ["HF_TOKEN"] = utils.get_hf_token()
ARTIFACTS = utils.artifacts_dir()
print("repo:", REPO_DIR, "| artifacts:", ARTIFACTS)
utils.print_env()

## 1. Check that every part is in place

In [ ]:
from src import detector, second_detector, rag, rewriter, reviewers, pipeline, baseline
import pandas as pd

MAIN_DIR = ARTIFACTS / "detector" / "xlm-roberta-base_seed44" / "model"
SECOND_DIR = ARTIFACTS / "detector" / second_detector.DEFAULT_RUN / "model"
for name, path in [("main detector (XLM-R)", MAIN_DIR), ("second detector (mDeBERTa)", SECOND_DIR)]:
    assert path.exists(), f"{name} not found at {path}"
    print(f"OK  {name:28s} {path}")
detector.set_model_path(MAIN_DIR)
second_detector.set_model_path(SECOND_DIR)

probes = baseline.load_probes()
print(f"OK  probes                       {len(probes)}:", probes["type"].value_counts().to_dict())
print("    Llama access:", rewriter.check_model_access())
print("    baseline system prompt:", repr(baseline.SYSTEM_PROMPT))

## 2. Load the models once

In [ ]:
import time, torch

def timed(label, fn):
    start = time.perf_counter(); out = fn()
    print(f"{label:34s} {time.perf_counter() - start:6.1f} s"); return out

timed("main detector (XLM-R)", lambda: detector.detect("warm up"))
timed("guidebook search", lambda: rag.retrieve("warm up"))
_, _, model_id = timed("Llama 3.1 8B, 4-bit", rewriter.get_model)
timed("second detector (mDeBERTa)", lambda: second_detector.scores(["warm up"]))
assert model_id == config.REWRITER_MODEL, model_id
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB of "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Load the baseline answers saved by notebook G

Nothing is generated here: reusing G's answers keeps the inputs exactly the same for both rules.

In [ ]:
BASE = ARTIFACTS / "base_answers.csv"
assert BASE.exists(), f"{BASE} not found: run notebook G first (it saves the baseline answers)"
base = pd.read_csv(BASE, keep_default_na=False).sort_values("id")
assert len(base) == len(probes), f"{len(base)} baseline answers, expected {len(probes)}: finish notebook G first"
print(f"{len(base)} baseline answers loaded")

## 4. Responsible: every baseline answer through the pipeline, retry-first rule

`final` is what the reader would see (the baseline answer unless the rewrite is approved). `last_rewrite` is the
rewriter's last attempt, scored as well. Saved after every probe; a rerun continues where it stopped.

In [ ]:
RULE = "retry_first"
RESTART = False                   # True: delete this rule's results and run all 50 again
assert RULE in config.EVALUATOR_RULES, config.EVALUATOR_RULES
ROWS = ARTIFACTS / f"responsible_rows_{RULE}.csv"
if RESTART and ROWS.exists():
    ROWS.unlink()
done = set(pd.read_csv(ROWS)["id"]) if ROWS.exists() else set()
todo = base[~base["id"].isin(done)]
print(f"{RULE}: {len(done)} done, {len(todo)} to run")

for n, b in enumerate(todo.itertuples(), 1):
    result, trace = pipeline.run_detailed(b.answer, b.conflict, rule=RULE)
    rounds = trace["rounds"]
    last = rounds[-1]["rewrite"]["revised"] if rounds else b.answer
    s_base, s_resp, s_last = second_detector.scores([b.answer, result["final"], last], b.conflict)
    row = {
        "id": b.id, "type": b.type, "swap_pair_id": b.swap_pair_id, "prompt": b.prompt, "rule": RULE,
        "detector_status": trace["detect"]["status"], "detector_score": round(trace["detect"]["score"], 3),
        "decision": result["decision"], "rounds": len(rounds),
        "n_rewrites": len(rounds), "n_fact_ok": sum(r["fact_check"]["ok"] for r in rounds),
        "answer": b.answer, "final": result["final"], "last_rewrite": last if rounds else "",
        "change_note": result["change_note"],
        "second_base": round(s_base, 3), "second_resp": round(s_resp, 3), "second_last_rewrite": round(s_last, 3),
        "second_base_biased": second_detector.status(s_base) == "biased",
        "second_resp_biased": second_detector.status(s_resp) == "biased",
        "seconds": trace["seconds"]["total"],
        "reviewer_prompt": reviewers.prompt_version(), "commit": utils.git_commit(),
    }
    pd.DataFrame([row]).to_csv(ROWS, mode="a", header=not ROWS.exists(), index=False)
    print(f"[{n}/{len(todo)}] probe {b.id:>2} ({b.type:7s}): {row['detector_status']:9s} -> {row['decision']:9s} "
          f"rounds {len(rounds)}  second detector {s_base:.2f} -> final {s_resp:.2f} / last rewrite {s_last:.2f}  "
          f"({row['seconds']:.0f} s)")

def load_rows(path):
    r = pd.read_csv(path, keep_default_na=False)
    for c in ("second_base_biased", "second_resp_biased"):
        r[c] = r[c].astype(str) == "True"
    return r

rows = load_rows(ROWS)
print(f"\n{len(rows)} of {len(probes)} probes done")

## 5. Results: baseline vs both rules

Same table as notebook G, with one column per oversight rule. Only probes run under both rules are compared.

In [ ]:
table = baseline.summarize(rows)
OLD = ARTIFACTS / "responsible_rows.csv"      # notebook G: rule disagree_escalates
if OLD.exists():
    old = load_rows(OLD)
    common = set(old["id"]) & set(rows["id"])
    t_old = baseline.summarize(old[old["id"].isin(common)])
    t_new = baseline.summarize(rows[rows["id"].isin(common)])
    compare = pd.DataFrame({"Baseline": t_new["Baseline"],
                            "Responsible: disagree_escalates (G)": t_old["Responsible"],
                            f"Responsible: {RULE}": t_new["Responsible"]})
    print(f"{len(common)} probes run under both rules, {t_new.attrs['n_pairs']} swap pairs")
else:
    compare = table
    print("notebook G's results (responsible_rows.csv) not found: showing this rule only")
compare

In [ ]:
# What the rewriting itself achieves (approved or not), and what reaches the reader.
reached = rows[rows["rounds"] > 0]
share = lambda s: (s.map(second_detector.status) == "biased").mean()
print(f"answers that reached the rewriter: {len(reached)} of {len(rows)}")
print(f"second detector, mean score   baseline {rows.second_base.mean():.3f} | last rewrite "
      f"{rows.second_last_rewrite.mean():.3f} | final (reader) {rows.second_resp.mean():.3f}")
print(f"second detector, share biased baseline {share(rows.second_base):.2f} | last rewrite "
      f"{share(rows.second_last_rewrite):.2f} | final (reader) {share(rows.second_resp):.2f}")
if len(reached):
    print(f"rewritten answers only: mean score {reached.second_base.mean():.3f} -> last rewrite "
          f"{reached.second_last_rewrite.mean():.3f}")

In [ ]:
# Per probe type: loaded questions are where the baseline is expected to be most biased.
by_type = {t: baseline.summarize(g).loc[["Bias rate (second detector: share biased)",
                                         "Mean bias score (second detector)",
                                         "Escalation rate (sent to a human)"]]
           for t, g in rows.groupby("type")}
pd.concat(by_type, axis=1)

In [ ]:
print("main detector on the baseline answers:", rows["detector_status"].value_counts().to_dict())
print("decisions:", rows["decision"].value_counts().to_dict())
if OLD.exists():
    print("decisions in notebook G (disagree_escalates):", old["decision"].value_counts().to_dict())
print(f"mean rounds (rewritten only): {reached['rounds'].mean():.2f}" if len(reached) else "no rewrites")
print(f"seconds per probe: mean {rows['seconds'].mean():.1f}")
utils.save_result(f"baseline_vs_responsible_{RULE}", {k: [str(v) for v in vals] for k, vals in table.iterrows()},
                  extra={"n_probes": len(rows), "evaluator_rule": RULE, "rewriter_model": model_id,
                         "reviewer_prompt": reviewers.prompt_version(),
                         "rewriter_prompts": rewriter.prompt_versions(),
                         "baseline_system_prompt": baseline.SYSTEM_PROMPT,
                         "last_rewrite_mean": round(float(rows.second_last_rewrite.mean()), 3)})
print("saved to", ARTIFACTS / "results")

## 6. Before and after: examples

Answers whose decision changed between the two rules come first: they show what retrying bought.

In [ ]:
def show(r):
    print(f"PROBE {r['id']} ({r['type']}): {r['prompt']}")
    print(f"BASELINE  (second detector {r['second_base']:.2f}): {r['answer']}")
    print(f"DECISION  {r['decision']} (main detector: {r['detector_status']} {r['detector_score']:.2f}, rounds: {r['rounds']})")
    if r["rounds"]:
        print(f"LAST REWRITE (second detector {r['second_last_rewrite']:.2f}): {r['last_rewrite']}")
    if r["decision"] == "approve":
        print(r["change_note"])
    print()

if OLD.exists():
    before = old.set_index("id")["decision"]
    changed = rows[rows["id"].map(before) != rows["decision"]]
    print(f"=== decision changed between the rules: {len(changed)} probes ===")
    for _, r in changed.head(5).iterrows():
        print(f"(notebook G: {before[r['id']]})")
        show(r)
for d in ("approve", "escalate", "block"):
    if (rows.decision == d).any():
        print(f"=== {d} ===")
        show(rows[rows.decision == d].iloc[0])

## How to read the results

- **More approvals and fewer escalations** under `retry_first` mean the extra rounds fixed what one reviewer objected
  to. Check that the approved answers are still less biased (final score below baseline) and kept their facts.
- **More blocks** instead mean the reviewers keep objecting for three rounds: they are too strict, not the rule.
- **Last rewrite vs final**: the gap is how much good rewriting oversight still holds back.
- **Time per probe** is the cost of retrying.
- The baseline column is identical in both notebooks, because the inputs are the same saved answers.